# Phase 1 & 2 — Raw Extraction and Landing Zone
**Apex Retail Intelligence | Celebal Technologies CEI'26 Major Project**

This notebook implements:
1. **Raw Zone** — ingest all inbound CSVs (historical + incremental, for customer/product/sales)
   as all-string schemas into `raw/<dataset>/<load_type>` on the pipeline Volume.
2. **Landing Zone** — convert Raw CSVs to Parquet (still all-string), then run the mandatory
   audit reconciliation against `audit_landing/*.csv` and emit a structured PASS/FAIL report.

The pipeline **halts** (raises an exception) if any table fails its audit check, per the
assignment's mandatory requirement.

## 0. Configuration & Widgets
Centralised path/catalog config so every notebook in the pipeline agrees on the same locations.
Update `CATALOG_ROOT` widget default to your Unity Catalog Volume path before first run.

In [0]:
dbutils.widgets.text("catalog_root", "/Volumes/apex_retail1/landing_zone/inbound_data", "Unity Catalog Volume Root")
dbutils.widgets.text("pipeline_root", "/Volumes/apex_retail1/pipeline/data", "Pipeline Storage Root (raw/landing/bronze)")
dbutils.widgets.text("run_date", "", "Run Date (yyyy-MM-dd, blank = today)")

CATALOG_ROOT = dbutils.widgets.get("catalog_root")   # where *source* CSVs + audit files were uploaded
PIPELINE_ROOT = dbutils.widgets.get("pipeline_root")  # where raw/ and landing/ artefacts get written
RUN_DATE = dbutils.widgets.get("run_date") or None

RAW_ROOT = f"{PIPELINE_ROOT}/raw"
LANDING_ROOT = f"{PIPELINE_ROOT}/landing"

print(f"CATALOG_ROOT  = {CATALOG_ROOT}")
print(f"PIPELINE_ROOT = {PIPELINE_ROOT}")
print(f"RAW_ROOT      = {RAW_ROOT}")
print(f"LANDING_ROOT  = {LANDING_ROOT}")

CATALOG_ROOT  = /Volumes/apex_retail1/landing_zone/inbound_data
PIPELINE_ROOT = /Volumes/apex_retail1/pipeline/data
RAW_ROOT      = /Volumes/apex_retail1/pipeline/data/raw
LANDING_ROOT  = /Volumes/apex_retail1/pipeline/data/landing


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException
import datetime as dt

RUN_TS = dt.datetime.now().isoformat(timespec="seconds")

# Dataset registry: logical name -> (source csv path on volume, load_type)
DATASET_REGISTRY = {
    "customer_historical":  (f"{CATALOG_ROOT}/historical_data/customer/customer_historical.csv",   "historical"),
    "customer_incremental": (f"{CATALOG_ROOT}/incremental_data/customer_incremental/customer_incremental.csv", "incremental"),
    "product_historical":   (f"{CATALOG_ROOT}/historical_data/product/product_historical.csv",     "historical"),
    "product_incremental":  (f"{CATALOG_ROOT}/incremental_data/product_incremental/product_incremental.csv",   "incremental"),
    "sales_historical":     (f"{CATALOG_ROOT}/historical_data/sales/sales_historical.csv",          "historical"),
    "sales_incremental":    (f"{CATALOG_ROOT}/incremental_data/sales_incremental/sales_incremental.csv",       "incremental"),
    
}

# audit_landing/<file>.csv declares expected row_count per table_name for this load
AUDIT_REGISTRY = {
    "customer_historical":  f"{CATALOG_ROOT}/audit_landing/customer_historical_audit.csv",
    "customer_incremental": f"{CATALOG_ROOT}/audit_landing/customer_incrementalaudit.csv",
    "product_historical":   f"{CATALOG_ROOT}/audit_landing/product_historical_audit.csv",
    "product_incremental":  f"{CATALOG_ROOT}/audit_landing/product_incrementalaudit.csv",
    "sales_historical":     f"{CATALOG_ROOT}/audit_landing/sales_historical_audit.csv",
    "sales_incremental":    f"{CATALOG_ROOT}/audit_landing/sales_incrementalaudit.csv",
}

# entity -> primary key, used later for logging only (no filtering happens until Silver)
ENTITY_OF = {
    "customer_historical": "customer", "customer_incremental": "customer",
    "product_historical": "product",   "product_incremental": "product",
    "sales_historical": "sales",       "sales_incremental": "sales",
}

## 1. Phase 1 — Raw Extraction
Every column is cast to **string** on the way in. Historical and incremental files are kept in
physically separate directories so downstream layers can treat them as distinct batches.

In [0]:
def ingest_to_raw(name, src_csv_path, load_type):
    """Read a source CSV, coerce every column to string, and persist under raw/<entity>/<load_type>/."""
    entity = ENTITY_OF[name]
    df = spark.read.option("header", True).option("inferSchema", False).csv(src_csv_path)
    df_string = df.select([F.col(c).cast("string").alias(c) for c in df.columns])
    dest = f"{RAW_ROOT}/{entity}/{load_type}"
    df_string.write.mode("overwrite").option("header", True).csv(dest)
    row_count = df_string.count()
    print(f"[RAW] {name:24s} -> {dest}  rows={row_count}")
    return row_count

raw_counts = {}
for name, (path, load_type) in DATASET_REGISTRY.items():
    raw_counts[name] = ingest_to_raw(name, path, load_type)

[RAW] customer_historical      -> /Volumes/apex_retail1/pipeline/data/raw/customer/historical  rows=1052
[RAW] customer_incremental     -> /Volumes/apex_retail1/pipeline/data/raw/customer/incremental  rows=1053
[RAW] product_historical       -> /Volumes/apex_retail1/pipeline/data/raw/product/historical  rows=1043
[RAW] product_incremental      -> /Volumes/apex_retail1/pipeline/data/raw/product/incremental  rows=1041
[RAW] sales_historical         -> /Volumes/apex_retail1/pipeline/data/raw/sales/historical  rows=1002
[RAW] sales_incremental        -> /Volumes/apex_retail1/pipeline/data/raw/sales/incremental  rows=1000


## 2. Phase 2 — Landing Conversion (CSV → Parquet)
Parquet is a compressed, columnar format that dramatically speeds up downstream scans
compared to re-parsing CSV on every read. Schema stays 100% string at this layer — type
casting is a Silver-layer concern, not a Landing-layer one.

In [0]:
def raw_to_landing(name, load_type):
    entity = ENTITY_OF[name]
    src = f"{RAW_ROOT}/{entity}/{load_type}"
    dest = f"{LANDING_ROOT}/{entity}/{load_type}"
    df = spark.read.option("header", True).csv(src)  # already all-string from Phase 1
    df.write.mode("overwrite").parquet(dest)
    row_count = spark.read.parquet(dest).count()
    print(f"[LANDING] {name:24s} -> {dest}  rows={row_count}")
    return row_count, dest

landing_counts = {}
landing_paths = {}
for name, (_, load_type) in DATASET_REGISTRY.items():
    cnt, path = raw_to_landing(name, load_type)
    landing_counts[name] = cnt
    landing_paths[name] = path

[LANDING] customer_historical      -> /Volumes/apex_retail1/pipeline/data/landing/customer/historical  rows=1052
[LANDING] customer_incremental     -> /Volumes/apex_retail1/pipeline/data/landing/customer/incremental  rows=1053
[LANDING] product_historical       -> /Volumes/apex_retail1/pipeline/data/landing/product/historical  rows=1043
[LANDING] product_incremental      -> /Volumes/apex_retail1/pipeline/data/landing/product/incremental  rows=1041
[LANDING] sales_historical         -> /Volumes/apex_retail1/pipeline/data/landing/sales/historical  rows=1002
[LANDING] sales_incremental        -> /Volumes/apex_retail1/pipeline/data/landing/sales/incremental  rows=1000


## 3. Mandatory Audit Reconciliation
Reads each `audit_landing/*.csv`, compares its declared `row_count` against the actual
landed Parquet row count, and builds a structured PASS/FAIL report. **Any FAIL halts the run.**

In [0]:
audit_results = []
for name, audit_path in AUDIT_REGISTRY.items():
    audit_df = spark.read.option("header", True).csv(audit_path)
    expected = int(audit_df.collect()[0]["row_count"])
    actual = landing_counts[name]
    status = "PASS" if expected == actual else "FAIL"
    audit_results.append((name, ENTITY_OF[name], expected, actual, status, RUN_TS))

audit_schema = ["table_name", "entity", "expected_row_count", "actual_row_count", "status", "run_timestamp"]
audit_report_df = spark.createDataFrame(audit_results, audit_schema)

display(audit_report_df)

table_name,entity,expected_row_count,actual_row_count,status,run_timestamp
customer_historical,customer,1052,1052,PASS,2026-08-14T08:12:41
customer_incremental,customer,1053,1053,PASS,2026-08-14T08:12:41
product_historical,product,1043,1043,PASS,2026-08-14T08:12:41
product_incremental,product,1041,1041,PASS,2026-08-14T08:12:41
sales_historical,sales,1002,1002,PASS,2026-08-14T08:12:41
sales_incremental,sales,1000,1000,PASS,2026-08-14T08:12:41


In [0]:
# Persist the audit report as a Delta table for full auditability across runs
(
    audit_report_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("apex_retail1.audit.landing_audit_log")
)

failed = [r for r in audit_results if r[4] == "FAIL"]
if failed:
    failed_names = ", ".join(r[0] for r in failed)
    raise Exception(
        f"LANDING AUDIT FAILED for: {failed_names}. "
        f"Pipeline halted per mandatory reconciliation requirement. "
        f"See table apex_retail.audit.landing_audit_log for details."
    )
else:
    print("✅ All Landing-zone audit checks PASSED. Safe to proceed to Bronze layer.")

✅ All Landing-zone audit checks PASSED. Safe to proceed to Bronze layer.


## 4. Hand-off to Bronze
Landing Parquet paths are written to a small control-table so `03_bronze_layer.py` can discover
them without hard-coding paths twice.

In [0]:
control_rows = [(name, ENTITY_OF[name], load_type, landing_paths[name], RUN_TS)
                 for name, (_, load_type) in DATASET_REGISTRY.items()]
control_df = spark.createDataFrame(control_rows, ["table_name", "entity", "load_type", "landing_path", "run_timestamp"])
control_df.write.format("delta").mode("overwrite").saveAsTable("apex_retail1.control.landing_paths")

print("Landing zone complete. Control table apex_retail.control.landing_paths updated.")

Landing zone complete. Control table apex_retail.control.landing_paths updated.
